
## Tiger Analytics Data Engineer Interview Questions (25+ LPA)

**Position:** Data Engineer
**Experience:** 4+ Years
**Application Process:** Referral

---

# Round 1 - Technical

1. Describe your project experience, including specific examples and your role in them.

2. Explain all concepts related to triggers in Azure Data Factory (ADF).

3. Implement upsert using pyspark

4. How do you restart a pipeline in ADF if it fails?

5. Explain how to deploy notebooks in prod using CI/CD.
6. '''
✅i. write PySpark code to get the desired output. 2nd DAY_SALE is the sale after the repeat sale (ex- 1st sell's 2NDDAY_SALE is the 3rd sale).

✅ii. Identify the Sale_date which has made 3rd highest sale for each product.
'''

---



## 2. Azure Data Factory (ADF) Trigger Types

Triggers in ADF automate pipeline execution. There are **three main types**:

### 1. **Schedule Trigger**
* **Purpose:** Executes pipelines on a **time-based schedule** (daily, hourly, weekly, etc.)
* **Use Case:** Regular ETL jobs, nightly data loads, periodic reporting
* **Key Features:**
  * Supports cron-like expressions for complex schedules
  * Can define start/end times
  * Frequency options: Minute, Hour, Day, Week, Month
* **Example:** Run a pipeline every day at 3 AM to process previous day's data

### 2. **Tumbling Window Trigger**
* **Purpose:** Executes pipelines for **fixed-size, non-overlapping time windows** with dependency support
* **Use Case:** Processing time-series data in consecutive windows, backfilling historical data
* **Key Features:**
  * Each window processes a specific time slice (e.g., hourly windows)
  * Supports **dependency chaining** between windows (current window can depend on previous window's success)
  * Can trigger **backfills** for historical windows
  * Guarantees each time window is processed exactly once
  * Passes window start/end times as parameters to the pipeline
* **Example:** Process each hour's data sequentially, ensuring Hour 2 only runs after Hour 1 succeeds
* **Key Difference from Schedule:** Schedule triggers run at specific times; tumbling window triggers process data FOR specific time ranges

### 3. **Event Trigger** (Storage Event Trigger)
* **Purpose:** Executes pipelines in response to **storage events** (file arrival/deletion in Azure Blob/ADLS Gen2)
* **Use Case:** Event-driven architectures, process files as soon as they arrive
* **Key Features:**
  * Monitors blob storage for events (BlobCreated, BlobDeleted)
  * Can filter by blob path patterns and size
  * Near real-time pipeline execution
  * Multiple pipelines can respond to the same event
* **Example:** Automatically start data ingestion when a new CSV file is uploaded to a specific container
* **Common Events:**
  * `Microsoft.Storage.BlobCreated` - New file created
  * `Microsoft.Storage.BlobDeleted` - File deleted

---

### Quick Comparison Table

| Trigger Type | Activation | Best For | Retry Logic |
|--------------|-----------|----------|-------------|
| **Schedule** | Time-based | Regular batch jobs | Manual retry |
| **Tumbling Window** | Time windows | Sequential time-series processing | Automatic retry per window |
| **Event** | File events | Real-time file processing | Event-driven, no built-in retry |

---

In [0]:
# 3. upsert implementation


from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "demo_table_one") 
 
#perform the UPSERT
 
(deltaTable.alias('orginal_table')
  .merge(df.alias('update_table'), "orginal_table.state_code = update_table.state_code and orginal_table.attom_id = update_table.attom_id")
  .whenNotMatchedInsertAll()
  .whenMatchedUpdateAll("orginal_table.sell_date < update_table.sell_date")
  .execute())


In [0]:
'''# 4. You can restart a failed pipeline in Azure Data Factory (ADF) by using the Monitor hub, with options to rerun from the beginning or rerun from the failed activity

In [0]:
# ✅i. write PySpark code to get the desired output. 2nd DAY_SALE is the sale after the repeat sale (ex- 1st sell's 2NDDAY_SALE is the 3rd sale).

# ✅ii. Identify the Sale_date which has made 3rd highest sale for each product. 


from pyspark.sql.functions import col, to_date, row_number, when, lead, lag,desc,asc, dense_rank, split, explode, collect_list,unix_timestamp,count, round
from pyspark.sql.window import Window

data = [('TV', '2016-11-27', 800),('TV', '2016-11-30', 900),('TV', '2016-12-29', 500),('TV', '2017-11-20', 400),('FRIDGE', '2016-10-11', 760),('FRIDGE', '2016-10-13', 400),('FRIDGE', '2016-11-27',460)]

schema = ['product','sale_date','amount']
df = spark.createDataFrame(data,schema)
df = df.withColumn('sale_date',to_date(col('sale_date'),'yyyy-MM-dd'))
df.show()

In [0]:

new_df = df.withColumn('NEXT_2NDDAY_SALE',lead('amount',2).over(Window.partitionBy('product').orderBy('sale_date'))).withColumn('PREVIOUS_2NDDAY_SALE',lag('amount',2).over(Window.partitionBy('product').orderBy('sale_date')))
new_df.orderBy(col('product').desc()).show()

In [0]:
final_df = df.withColumn('rnk',dense_rank().over(Window.partitionBy('product').orderBy(col('amount').desc()))).filter('rnk == 3').select('product','sale_date')
final_df.show()

# Round 2 - Advanced Technical

1. Explain Databricks Delta Live Tables (DLT). When do you use batch vs. streaming?

2. How do you ensure data quality and integrity in your ETL processes?

3. Describe the architecture and implementation details of your recent project.

4. How do you handle schema evolution in your data pipelines?

5. What strategies do you use to optimize Spark jobs for large-scale datasets?

6. Explain data partitioning and why it's beneficial.

---



In [0]:
# 1. Explain Databricks Delta Live Tables (DLT). When do you use batch vs. streaming?

# Lakeflow Declarative Pipelines (formerly Delta Live Tables) enable declarative ETL pipelines.
# Use batch (materialized views) for periodic, full data processing; use streaming tables for real-time, incremental data.

from pyspark import pipelines as dp

# Materialized View (Batch): Precomputes results, ideal for batch ETL and analytics.
@dp.table(
    name="customer_mv",
    comment="Materialized view for batch processing"
)
def customer_mv():
    return spark.read.table("source.customers")

# Streaming Table: Processes data incrementally as it arrives, ideal for real-time ingestion and low-latency transformations.
@dp.table(
    name="customer_streaming",
    comment="Streaming table for real-time processing"
)
def customer_streaming():
    return spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .load("/mnt/data/customers/")

In [0]:
# 2. How do you ensure data quality and integrity in your ETL processes?

# I ensure data quality and integrity by implementing validation checks throughout the ETL process. At ingestion, I validate file formats, schemas, and mandatory fields. During transformation, I enforce data types, remove duplicates, handle null values, and apply business validation rules. Before loading, I reconcile source and target row counts and verify key metrics such as aggregates and distinct counts. I also maintain detailed logging and auditing, isolate invalid records in a quarantine table, and implement retry mechanisms for transient failures. When working with Delta Lake, I leverage ACID transactions and schema enforcement to ensure reliable and consistent data loads. This layered approach helps deliver accurate, consistent, and trustworthy data for downstream reporting and analytics.

In [0]:
# 4. How do you handle schema evolution in your data pipelines?

# Lakeflow Declarative Pipelines automatically handle schema evolution.
# No explicit configuration is needed; pipelines adapt to schema changes in sources.

from pyspark import pipelines as dp

@dp.table(
    name="evolving_customers",
    comment="Handles schema evolution automatically"
)
def evolving_customers():
    return spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .load("/mnt/data/customers/")

# When new columns are added to the source, Lakeflow Declarative Pipelines detects and updates downstream tables.
# For batch sources, use spark.read.table(...) instead of spark.readStream.

5. When optimizing Spark jobs for large-scale datasets, I focus on minimizing data movement, improving parallelism, and making efficient use of cluster resources. Here's how I typically approach it:

### 1. Choose the Right File Format

* Use **Parquet** or **Delta Lake** instead of CSV or JSON.
* These formats support:

  * Columnar storage
  * Predicate pushdown
  * Compression
  * Faster reads

---

### 2. Partition Data Effectively

* Partition tables based on frequently filtered columns such as:

  * Date
  * Region
  * Customer ID (if appropriate)
* Avoid over-partitioning, which creates many small files.

```python
df.write.partitionBy("year", "month").parquet(path)
```

---

### 3. Optimize Shuffle Operations

Shuffles are one of the biggest performance bottlenecks.

I reduce shuffles by:

* Filtering data before joins
* Selecting only required columns
* Using `reduceByKey` instead of `groupByKey`
* Avoiding unnecessary `distinct()` and `orderBy()`

---

### 4. Use Broadcast Joins

If one table is small, broadcast it to avoid expensive shuffles.

```python
from pyspark.sql.functions import broadcast

result = large_df.join(broadcast(small_df), "id")
```

This significantly improves join performance.

---

### 5. Handle Data Skew

Data skew causes some tasks to process much more data than others.

Solutions include:

* Salting skewed keys
* Adaptive Query Execution (AQE)
* Repartitioning skewed data
* Broadcast joins when applicable

---

### 6. Cache or Persist Reused Data

Cache DataFrames that are reused multiple times.

```python
df.cache()
```

Or use:

```python
df.persist(StorageLevel.MEMORY_AND_DISK)
```

Avoid caching data that's only used once.

---

### 7. Tune the Number of Partitions

Default partition counts are often not optimal.

```python
spark.conf.set("spark.sql.shuffle.partitions", 200)
```

* Increase partitions for large datasets.
* Reduce partitions for smaller workloads.

---

### 8. Repartition vs. Coalesce

Use the appropriate method depending on your goal.

* **repartition()**

  * Performs a full shuffle
  * Good for increasing partitions

* **coalesce()**

  * Avoids shuffle when reducing partitions
  * Better before writing output

```python
df.coalesce(10)
```

---

### 9. Enable Adaptive Query Execution (AQE)

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
```

AQE automatically:

* Optimizes joins
* Handles skew
* Dynamically adjusts shuffle partitions

---

### 10. Filter Early and Select Required Columns

Instead of:

```python
df.select("*")
```

Use:

```python
df.select("id", "name", "salary")
```

Push filters as early as possible:

```python
df.filter(df.status == "Active")
```

This reduces the amount of data processed.

---

### 11. Avoid User-Defined Functions (UDFs) When Possible

Built-in Spark SQL functions are much faster because they're optimized by the Catalyst optimizer.

Instead of:

```python
udf(...)
```

Prefer:

```python
from pyspark.sql.functions import upper
```

---

### 12. Optimize Memory Usage

Configure executors appropriately:

```text
spark.executor.memory
spark.executor.cores
spark.driver.memory
spark.memory.fraction
```

Avoid allocating too many cores per executor, as it can increase garbage collection pauses.

---

### 13. Monitor Spark UI

I regularly use the Spark UI to identify bottlenecks by checking:

* Long-running stages
* Shuffle read/write sizes
* Task skew
* Spill to disk
* Executor utilization
* Garbage collection time

This helps pinpoint where optimization is needed.

---

### 14. Optimize Writes

* Write larger files instead of many small files.
* Use compression (Snappy is common for Parquet).
* Compact small files if necessary.

---

### 15. Leverage Delta Lake Optimizations (if using Delta)

For Delta tables, I use:

* `OPTIMIZE` to compact files
* `ZORDER BY` for faster queries on commonly filtered columns
* `VACUUM` to clean up obsolete files

---

## Real-world Example

In one project, I worked on a pipeline processing approximately **2 TB of daily transaction data**. Initially, the job took around **3.5 hours**.

We improved performance by:

* Converting CSV files to Parquet
* Broadcasting small dimension tables
* Enabling Adaptive Query Execution
* Repartitioning based on business keys
* Eliminating unnecessary shuffles
* Filtering data before joins
* Tuning shuffle partitions and executor memory

These changes reduced the runtime to **about 1 hour**, while also lowering cluster resource usage.

### Interview Summary

A concise response you could give in an interview:

> "When optimizing Spark jobs, I first analyze the execution plan and Spark UI to identify bottlenecks. I reduce shuffles by filtering early and selecting only necessary columns, use broadcast joins for small tables, handle data skew with repartitioning or AQE, cache reused DataFrames, tune partition counts, and prefer Parquet or Delta formats. I also avoid unnecessary UDFs, optimize executor memory, and use Delta optimizations like `OPTIMIZE` and `ZORDER` where applicable. These practices help improve performance and reduce resource consumption on large-scale datasets."


In [0]:
# 6. Explain data partitioning and why it's beneficial.

# Data partitioning is the process of dividing a large dataset into smaller partitions so Spark can process them in parallel across multiple executors. It improves performance by enabling parallel processing, maximizing CPU utilization, reducing query execution time through partition pruning, minimizing network shuffles during joins, and providing better scalability and fault tolerance. In Spark, we use methods like repartition() and coalesce() to manage processing partitions, and partitionBy() when writing data to formats like Parquet or Delta Lake. Choosing an appropriate partitioning strategy based on frequently filtered or joined columns is essential for optimizing large-scale Spark jobs.

# Round 3 - Managerial / Behavioral

1. Tell me about a time you handled a production issue under pressure. How did you manage it?

2. How do you explain technical solutions to non-technical clients?

3. Imagine a client has unrealistic expectations on delivery timelines. How would you handle it?

4. Describe a situation where you worked with multiple teams having conflicting priorities. How did you manage it?
